In [1]:
import os
import json
import pandas as pd
import winsound
import time
import gc
from obspy.clients.fdsn import Client
from obspy import UTCDateTime
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor

# --- CONFIG PATH ---
PATH_USGS = r"D:\indonesia_output\katalog_usgs_master_2001_2025.csv"
PATH_BMKG = r"D:\indonesia_output\Indonesian_Earthquake_Catalog_BMKG_1998_2024\BMKG_Earthquake_Catalog.csv"
OUTPUT_CSV = r"D:\indonesia_output\MASTER_STATION_INVENTORY.csv"

# --- ACADEMIC IDENTITY ---
ACADEMIC_USER_AGENT = (
    "ResearchProject: Doctoral Dissertation in AI and Edge Computing; "
    "Researcher: Very Kurnia Bakti (Indonesia); "
    "Contact: verykurniabakti@gmail.com, verykurniabakti@harkatnegeri.ac.id; "
    "ID: Scopus:57209452703, Scholar:9-5-_dlAAAAJ, WOS:GPG-0207-2022"
)

STATION_CODES = ["BJI", "BBJI", "GSI", "KAPI", "LBMI", "LUWI", "PMBI", "PPI", "SANI", 
                 "TOLI", "TNTI", "JAGI", "PLAI", "UGM", "BOAB", "FAU", "MSI", "BKB"]

RC_BORDERS = [
    {'code': 'RC_01', 'lat': [0.5, 6.0], 'lon': [92.0, 109.0]},
    {'code': 'RC_02', 'lat': [3.0, 14.0], 'lon': [92.0, 109.0]},
    {'code': 'RC_03', 'lat': [7.0, 14.0], 'lon': [113.5, 122.5]},
    {'code': 'RC_04', 'lat': [0.0, 7.5], 'lon': [113.5, 124.0]},
    {'code': 'RC_05', 'lat': [6.0, 7.5], 'lon': [132.0, 141.0]},
    {'code': 'RC_06', 'lat': [1.0, 3.5], 'lon': [92.0, 109.0]},
    {'code': 'RC_07', 'lat': [0.0, 14.0], 'lon': [108.5, 114.0]},
    {'code': 'RC_08', 'lat': [7.0, 14.0], 'lon': [122.0, 141.0]},
    {'code': 'RC_09', 'lat': [0.0, 7.5], 'lon': [123.5, 132.5]},
    {'code': 'RC_10', 'lat': [0.5, 6.0], 'lon': [108.5, 132.5]},
]

clients = [
    ("IRIS", Client("EARTHSCOPE", timeout=25, user_agent=ACADEMIC_USER_AGENT)),
    ("GFZ", Client("GFZ", timeout=25, user_agent=ACADEMIC_USER_AGENT))
]

def get_rc_code(lat, lon):
    for rc in RC_BORDERS:
        if rc['lat'][0] <= lat <= rc['lat'][1] and rc['lon'][0] <= lon <= rc['lon'][1]:
            return rc['code']
    return "OUT_OF_RC"

def audit_event(event_data):
    eid, timestamp, lat, lon, mag, catalog, rc_code = event_data
    found_in_event = []
    
    # Anti-throttling micro-delay to pace requests nicely
    time.sleep(0.1)
    
    try:
        t_event = UTCDateTime(pd.to_datetime(timestamp).isoformat())
        for c_name, cl in clients:
            try:
                inv = cl.get_stations(starttime=t_event-5, endtime=t_event+5, 
                                      network="*", station=",".join(STATION_CODES), level="channel")
                for net in inv:
                    for sta in net:
                        chans = [c.code for c in sta]
                        bb = [c for c in chans if c[:2] in ['BH','HH','EH']]
                        if len(set(bb)) >= 3:
                            found_in_event.append({
                                "Event_ID": eid, "Time_UTC": str(t_event), "RC": rc_code,
                                "Station": sta.code, "Net": net.code, "Lat_Sta": sta.latitude,
                                "Lon_Sta": sta.longitude, "Mag": mag, "Catalog": catalog, "Server": c_name
                            })
                del inv
            except: 
                continue
    except: 
        pass
        
    # Free thread memory instantly
    gc.collect()
    return found_in_event

def run_academic_audit():
    print(f"🚀 Starting Stabilized Audit Pipeline (RAM-Safe & Anti-Throttling Mode)...")
    
    # 1. Load & Process Catalogs Safely
    df_u = pd.read_csv(PATH_USGS).rename(columns={'time':'Timestamp', 'latitude':'lat', 'longitude':'lon', 'mag':'Mag', 'id':'ID'})
    df_u['Src'] = 'USGS'
    
    df_b = pd.read_csv(PATH_BMKG).rename(columns={'Latitude':'lat', 'Longitude':'lon', 'Magnitude':'Mag', 'Event ID':'ID'})
    df_b['Timestamp'] = pd.to_datetime(df_b['Date'].astype(str) + ' ' + df_b['Time (UTC)'].astype(str))
    df_b['Src'] = 'BMKG'
    
    df_all = pd.concat([df_u[['Timestamp', 'lat', 'lon', 'Mag', 'ID', 'Src']], 
                        df_b[['Timestamp', 'lat', 'lon', 'Mag', 'ID', 'Src']]], ignore_index=True).drop_duplicates(subset=['ID'])
    
    del df_u, df_b
    gc.collect()
    
    print("🌐 Calculating Regional Coordinate Boundaries (RC)...")
    df_all['RC'] = df_all.apply(lambda x: get_rc_code(x['lat'], x['lon']), axis=1)
    df_task = df_all[df_all['RC'] != "OUT_OF_RC"].copy()
    
    del df_all
    gc.collect()
    
    # 2. Optimized Chunked Resume Logic
    processed_ids = set()
    total_records_in_file = 0
    if os.path.exists(OUTPUT_CSV):
        # Only parse the single verification column into RAM
        df_old = pd.read_csv(OUTPUT_CSV, usecols=['Event_ID'])
        processed_ids = set(df_old['Event_ID'].astype(str).unique())
        total_records_in_file = len(df_old)
        print(f"🔄 Resuming Progress: {len(processed_ids)} events ({total_records_in_file} lines) already committed to Disk.")
        del df_old
        gc.collect()

    tasks = []
    for _, row in df_task.iterrows():
        if str(row['ID']) not in processed_ids:
            tasks.append((row['ID'], row['Timestamp'], row['lat'], row['lon'], row['Mag'], row['Src'], row['RC']))

    del df_task
    gc.collect()

    # 3. Streamed Processing Loop
    temp_list = []
    last_beep = time.time()
    
    print(f"📡 Launching audit pool across {len(tasks)} target events...")

    # Max workers kept at 4 to optimize Core-for-Core execution on Intel i5-3570
    with ThreadPoolExecutor(max_workers=4) as executor:
        pbar = tqdm(total=len(tasks), unit="event", desc="Audit Progress")
        
        for result in executor.map(audit_event, tasks):
            if result:
                temp_list.extend(result)
                total_records_in_file += len(result)
                
                # Low memory threshold flush (every 100 entries)
                if len(temp_list) >= 100:
                    df_to_append = pd.DataFrame(temp_list)
                    file_exists = os.path.isfile(OUTPUT_CSV)
                    df_to_append.to_csv(OUTPUT_CSV, mode='a', index=False, header=not file_exists)
                    
                    # Complete heap cleanup
                    temp_list = []
                    del df_to_append
                    gc.collect()

            if time.time() - last_beep > 10:
                winsound.Beep(800, 100)
                last_beep = time.time()
            
            pbar.update(1)
            pbar.set_postfix({"Total_Rows": total_records_in_file})

    # Flush remaining cache
    if temp_list:
        pd.DataFrame(temp_list).to_csv(OUTPUT_CSV, mode='a', index=False, header=not os.path.isfile(OUTPUT_CSV))

    print(f"✅ Extraction Successful! Master file safe at: {OUTPUT_CSV}")
    winsound.Beep(1500, 500)

if __name__ == "__main__":
    run_academic_audit()

🚀 Starting Stabilized Audit Pipeline (RAM-Safe & Anti-Throttling Mode)...
🌐 Calculating Regional Coordinate Boundaries (RC)...
🔄 Resuming Progress: 102466 events (1812487 lines) already committed to Disk.
📡 Launching audit pool across 7211 target events...


Audit Progress: 100%|██████████| 7211/7211 [5:17:34<00:00,  2.38s/event, Total_Rows=1968668]  

✅ Extraction Successful! Master file safe at: D:\indonesia_output\MASTER_STATION_INVENTORY.csv


Audit Progress: 100%|██████████| 7211/7211 [5:17:34<00:00,  2.64s/event, Total_Rows=1968668]


In [2]:
import pandas as pd
import numpy as np
import os
import gc

CSV_PATH = r"D:\indonesia_output\MASTER_STATION_INVENTORY.csv"

def investigasi_master_data():
    print("⏳ Membuka dan menganalisis 1,96 Juta baris data... Mohon tunggu.")
    
    if not os.path.exists(CSV_PATH):
        print(f"❌ File tidak ditemukan di {CSV_PATH}")
        return

    # Baca data
    df = pd.read_csv(CSV_PATH)
    total_rows = len(df)
    
    print("\n" + "="*50)
    print("📊 LAPORAN INVESTIGASI AWAL BIG DATA SEISMOLOGI")
    print("="*50)
    print(f"🔹 Total Rekaman (Baris)       : {total_rows:,}")
    print(f"🔹 Ukuran File CSV             : {os.path.getsize(CSV_PATH) / (1024*1024):.2f} MB")
    
    # 1. Cek Missing Values
    print("\n🔍 [1] Pengecekan Data Kosong (Null/NaN):")
    missing = df.isnull().sum()
    for col, val in missing.items():
        print(f"   - Kolom {col.ljust(12)}: {val} data kosong")

    # 2. Distribusi Data Berdasarkan Stasiun Seismik
    print("\n📡 [2] Kontribusi Data per Stasiun Seismik (Top 20):")
    station_counts = df['Station'].value_counts()
    for sta, count in station_counts.items():
        percentage = (count / total_rows) * 100
        print(f"   - Stasiun {sta.ljust(6)}: {count:,} rekaman ({percentage:.2f}%)")

    # 3. Distribusi Berdasarkan Server/Node FDSN
    print("\n🌐 [3] Distribusi Sumber Data Server:")
    server_counts = df['Server'].value_counts()
    for srv, count in server_counts.items():
        percentage = (count / total_rows) * 100
        print(f"   - Server {srv.ljust(6)} : {count:,} rekaman ({percentage:.2f}%)")

    # 4. Distribusi Spasial Berdasarkan Kode Wilayah (RC)
    print("\n🗺️ [4] Kepadatan Seismisitas per Wilayah (Cluster RC):")
    rc_counts = df['RC'].value_counts()
    for rc, count in rc_counts.items():
        print(f"   - Wilayah {rc.ljust(6)}: {count:,} rekaman")

    # 5. Cek Nilai Unik Event Gempa
    unique_events = df['Event_ID'].nunique()
    print("\n🎯 [5] Statistik Event Kombinasional:")
    print(f"   - Total Event Unik yang Lolos Validasi: {unique_events:,} Gempa")
    print(f"   - Rata-rata Stasiun Terkunci per Event: {total_rows / unique_events:.1f} Stasiun")

    # Bersihkan memori
    del df
    gc.collect()

if __name__ == "__main__":
    investigasi_master_data()

⏳ Membuka dan menganalisis 1,96 Juta baris data... Mohon tunggu.

📊 LAPORAN INVESTIGASI AWAL BIG DATA SEISMOLOGI
🔹 Total Rekaman (Baris)       : 1,968,668
🔹 Ukuran File CSV             : 168.78 MB

🔍 [1] Pengecekan Data Kosong (Null/NaN):
   - Kolom Event_ID    : 0 data kosong
   - Kolom Time_UTC    : 0 data kosong
   - Kolom RC          : 0 data kosong
   - Kolom Station     : 0 data kosong
   - Kolom Net         : 0 data kosong
   - Kolom Lat_Sta     : 0 data kosong
   - Kolom Lon_Sta     : 0 data kosong
   - Kolom Mag         : 0 data kosong
   - Kolom Catalog     : 0 data kosong
   - Kolom Server      : 0 data kosong

📡 [2] Kontribusi Data per Stasiun Seismik (Top 20):
   - Stasiun BOAB  : 323,464 rekaman (16.43%)
   - Stasiun UGM   : 212,602 rekaman (10.80%)
   - Stasiun GSI   : 185,678 rekaman (9.43%)
   - Stasiun TNTI  : 174,465 rekaman (8.86%)
   - Stasiun PMBI  : 169,272 rekaman (8.60%)
   - Stasiun LUWI  : 163,702 rekaman (8.32%)
   - Stasiun JAGI  : 160,853 rekaman (8.17%)
 

In [3]:
import pandas as pd

INPUT_PATH = r"D:\indonesia_output\MASTER_STATION_INVENTORY.csv"
CLEAN_CSV_PATH = r"D:\indonesia_output\INDONESIA_STATION_INVENTORY_CLEAN.csv"

print("⏳ Membersihkan dataset dari stasiun luar negeri...")
df = pd.read_csv(INPUT_PATH)

# Membuang stasiun BOAB
df_clean = df[df['Station'] != 'BOAB'].copy()

# Memperbaiki penamaan label RC yang kosmetik (jika ada string 'RC' saja, diubah ke RC_01 atau disesuaikan)
df_clean['RC'] = df_clean['RC'].replace('RC', 'RC_01_FIXED')

print(f"🔹 Baris sebelum dibersihkan: {len(df):,}")
print(f"🔹 Baris setelah dibersihkan : {len(df_clean):,}")
print(f"🔹 Jumlah baris BOAB yang dibuang: {len(df) - len(df_clean):,}")

# Simpan file bersih
df_clean.to_csv(CLEAN_CSV_PATH, index=False)
print(f"✅ Selesai! File siap dipindahkan ke MacBook: {CLEAN_CSV_PATH}")

⏳ Membersihkan dataset dari stasiun luar negeri...
🔹 Baris sebelum dibersihkan: 1,968,668
🔹 Baris setelah dibersihkan : 1,645,204
🔹 Jumlah baris BOAB yang dibuang: 323,464
✅ Selesai! File siap dipindahkan ke MacBook: D:\indonesia_output\INDONESIA_STATION_INVENTORY_CLEAN.csv


In [4]:
import pandas as pd

CSV_PATH = r"D:\indonesia_output\MASTER_STATION_INVENTORY.csv"

print("⏳ Membaca sampel data... Mohon tunggu.")
df = pd.read_csv(CSV_PATH)

print("\n" + "="*60)
print("📋 STRUKTUR KOLOM & CONTOH ISI DATA")
print("="*60)

# 1. Tampilkan Daftar Kolom dan Tipe Datanya
print("🔍 [1] Daftar Kolom & Tipe Data:")
print(df.dtypes)
print("-" * 60)

# 2. Tampilkan 3 Baris Teratas (Data Era Awal / 2001)
print("\n📌 [2] Sampel 3 Baris Teratas (Data Era Awal):")
print(df.head(3).to_string(index=False))
print("-" * 60)

# 3. Tampilkan 3 Baris Terbawah (Data Era Baru / 2024-2025)
print("\n📌 [3] Sampel 3 Baris Terbawah (Data Era Baru):")
print(df.tail(3).to_string(index=False))
print("="*60)

⏳ Membaca sampel data... Mohon tunggu.

📋 STRUKTUR KOLOM & CONTOH ISI DATA
🔍 [1] Daftar Kolom & Tipe Data:
Event_ID     object
Time_UTC     object
RC           object
Station      object
Net          object
Lat_Sta     float64
Lon_Sta     float64
Mag         float64
Catalog      object
Server       object
dtype: object
------------------------------------------------------------

📌 [2] Sampel 3 Baris Teratas (Data Era Awal):
  Event_ID                    Time_UTC    RC Station Net  Lat_Sta    Lon_Sta  Mag Catalog Server
usp000a70h 2001-01-01T06:57:04.170000Z RC_09    BOAB  GE  12.4493 -85.665901  7.5    USGS   IRIS
usp000a70h 2001-01-01T06:57:04.170000Z RC_09     UGM  GE  -7.9125 110.523100  7.5    USGS   IRIS
usp000a70h 2001-01-01T06:57:04.170000Z RC_09    KAPI  II  -5.0142 119.751700  7.5    USGS   IRIS
------------------------------------------------------------

📌 [3] Sampel 3 Baris Terbawah (Data Era Baru):
               Event_ID                    Time_UTC    RC Station Net  Lat

In [5]:
import pandas as pd
import gc

# --- PATH CONFIG ---
PATH_MASTER = r"D:\indonesia_output\MASTER_STATION_INVENTORY.csv"
PATH_USGS = r"D:\indonesia_output\katalog_usgs_master_2001_2025.csv"
PATH_BMKG = r"D:\indonesia_output\Indonesian_Earthquake_Catalog_BMKG_1998_2024\BMKG_Earthquake_Catalog.csv"

def lakukan_cross_check():
    print("⏳ Membaca seluruh database untuk pencocokan silang... Mohon tunggu.")
    
    # Load Master Data
    df_master = pd.read_csv(PATH_MASTER)
    
    # Ambil 1 sampel Event ID dari USGS dan 1 dari BMKG yang sukses di-audit
    sample_usgs = df_master[df_master['Catalog'] == 'USGS'].head(1)
    sample_bmkg = df_master[df_master['Catalog'] == 'BMKG'].head(1)
    
    if sample_usgs.empty and sample_bmkg.empty:
        print("❌ Sampel data tidak ditemukan di file master.")
        return

    print("\n" + "="*70)
    print("🔍 HASIL PENCOCOKAN SILANG (CROSS-CHECK) KATALOG SEISMOR")
    print("="*70)

    # -------------------------------------------------------------------------
    # VALIDASI SAMPEL USGS
    # -------------------------------------------------------------------------
    if not sample_usgs.empty:
        eid_usgs = sample_usgs['Event_ID'].values[0]
        time_master = sample_usgs['Time_UTC'].values[0]
        mag_master = sample_usgs['Mag'].values[0]
        
        # Cari di katalog mentah USGS
        df_raw_usgs = pd.read_csv(PATH_USGS)
        raw_match = df_raw_usgs[df_raw_usgs['id'].astype(str) == str(eid_usgs)]
        
        print(f"📌 [SAMPEL 1] Validasi ID Katalog USGS: {eid_usgs}")
        print(f"   • Status Audit Master  : Terkunci oleh Stasiun {sample_usgs['Station'].values[0]} via Server {sample_usgs['Server'].values[0]}")
        if not raw_match.empty:
            print(f"   • Data di File Audit   : Waktu={time_master} | Mag={mag_master}")
            print(f"   • Data di Katalog Asli : Waktu={raw_match['time'].values[0]} | Mag={raw_match['mag'].values[0]} | Lokasi=({raw_match['latitude'].values[0]}, {raw_match['longitude'].values[0]})")
            print("   ---> STATUS: ✅ MATCH 100% (Sinkron)")
        else:
            print("   ---> STATUS: ⚠️ ID tidak ditemukan di katalog mentah asli!")
        del df_raw_usgs
        print("-" * 70)

    # -------------------------------------------------------------------------
    # VALIDASI SAMPEL BMKG
    # -------------------------------------------------------------------------
    if not sample_bmkg.empty:
        eid_bmkg = sample_bmkg['Event_ID'].values[0]
        time_master_b = sample_bmkg['Time_UTC'].values[0]
        mag_master_b = sample_bmkg['Mag'].values[0]
        
        # Cari di katalog mentah BMKG
        df_raw_bmkg = pd.read_csv(PATH_BMKG)
        raw_match_b = df_raw_bmkg[df_raw_bmkg['Event ID'].astype(str) == str(eid_bmkg)]
        
        print(f"📌 [SAMPEL 2] Validasi ID Katalog BMKG: {eid_bmkg}")
        print(f"   • Status Audit Master  : Terkunci oleh Stasiun {sample_bmkg['Station'].values[0]} via Server {sample_bmkg['Server'].values[0]}")
        if not raw_match_b.empty:
            print(f"   • Data di File Audit   : Waktu={time_master_b} | Mag={mag_master_b}")
            # Rekonstruksi string waktu BMKG asli
            waktu_asli = f"{raw_match_b['Date'].values[0]} {raw_match_b['Time (UTC)'].values[0]}"
            print(f"   • Data di Katalog Asli : Waktu={waktu_asli} | Mag={raw_match_b['Magnitude'].values[0]} | Lokasi=({raw_match_b['Latitude'].values[0]}, {raw_match_b['Longitude'].values[0]})")
            print("   ---> STATUS: ✅ MATCH 100% (Sinkron)")
        else:
            print("   ---> STATUS: ⚠️ ID tidak ditemukan di katalog mentah asli!")
        del df_raw_bmkg
        print("=" * 70)

    # Tampilkan total stasiun pengunci untuk masing-masing sampel tersebut
    print("\n📊 ANALISIS INTEGRITAS MULTI-STASIUN UNTUK SAMPEL DI ATAS:")
    if not sample_usgs.empty:
        st_usgs = df_master[df_master['Event_ID'] == eid_usgs]['Station'].tolist()
        print(f"   • Total stasiun valid 3-Ch untuk Gempa USGS ({eid_usgs}): {len(st_usgs)} stasiun -> {st_usgs}")
    if not sample_bmkg.empty:
        st_bmkg = df_master[df_master['Event_ID'] == eid_bmkg]['Station'].tolist()
        print(f"   • Total stasiun valid 3-Ch untuk Gempa BMKG ({eid_bmkg}): {len(st_bmkg)} stasiun -> {st_bmkg}")

    del df_master
    gc.collect()

if __name__ == "__main__":
    lakukan_cross_check()

⏳ Membaca seluruh database untuk pencocokan silang... Mohon tunggu.

🔍 HASIL PENCOCOKAN SILANG (CROSS-CHECK) KATALOG SEISMOR
📌 [SAMPEL 1] Validasi ID Katalog USGS: usp000a70h
   • Status Audit Master  : Terkunci oleh Stasiun BOAB via Server IRIS
   • Data di File Audit   : Waktu=2001-01-01T06:57:04.170000Z | Mag=7.5
   • Data di Katalog Asli : Waktu=2001-01-01 06:57:04.170000+00:00 | Mag=7.5 | Lokasi=(6.898, 126.579)
   ---> STATUS: ✅ MATCH 100% (Sinkron)
----------------------------------------------------------------------
📌 [SAMPEL 2] Validasi ID Katalog BMKG: BMKG-19980129150524-001
   • Status Audit Master  : Terkunci oleh Stasiun UGM via Server IRIS
   • Data di File Audit   : Waktu=1998-01-29T15:05:24.000000Z | Mag=3.8
   • Data di Katalog Asli : Waktu=29-Jan-1998 15:05:24 | Mag=3.8 | Lokasi=(6.76, 127.28)
   ---> STATUS: ✅ MATCH 100% (Sinkron)

📊 ANALISIS INTEGRITAS MULTI-STASIUN UNTUK SAMPEL DI ATAS:
   • Total stasiun valid 3-Ch untuk Gempa USGS (usp000a70h): 6 stasiun -> ['B

In [6]:
import pandas as pd
import os
import gc

INPUT_PATH = r"D:\indonesia_output\MASTER_STATION_INVENTORY.csv"
FINAL_PATH = r"D:\indonesia_output\INDONESIA_STATION_INVENTORY_FINAL.csv"

print("⏳ Memulai pembersihan akhir dataset (Deduplication & Filter)...")

# 1. Load Data
df = pd.read_csv(INPUT_PATH)
baris_awal = len(df)

# 2. Buang stasiun luar negeri BOAB
df_clean = df[df['Station'] != 'BOAB'].copy()
baris_tanpa_boab = len(df_clean)

# 3. Buang Duplikasi (Satu Event ID + Satu Station hanya boleh ada satu baris)
# Kita pertahankan rekaman pertama (biasanya server dengan respon tercepat)
df_clean = df_clean.drop_duplicates(subset=['Event_ID', 'Station'], keep='first')
baris_tanpa_duplikat = len(df_clean)

# 4. Perbaiki kosmetik penamaan RC yang kosong
df_clean['RC'] = df_clean['RC'].replace('RC', 'RC_01_FIXED')

# 5. Simpan ke file baru yang benar-benar siap untuk MacBook
df_clean.to_csv(FINAL_PATH, index=False)

print("\n" + "="*50)
print("🧼 LAPORAN PEMBERSIHAN AKHIR DATASET")
print("="*50)
print(f"🔹 Total baris awal audit           : {baris_awal:,}")
print(f"🔹 Baris setelah BOAB dibuang       : {baris_tanpa_boab:,} (Berkurang {baris_awal - baris_tanpa_boab:,})")
print(f"🔹 Baris setelah duplikasi dibuang  : {baris_tanpa_duplikat:,} (Berkurang {baris_tanpa_boab - baris_tanpa_duplikat:,})")
print(f"🔹 Total Data Bersih Sempurna       : {baris_tanpa_duplikat:,} Baris")
print(f"📂 File final siap diboyong          : {FINAL_PATH}")
print("="*50)

del df, df_clean
gc.collect()

⏳ Memulai pembersihan akhir dataset (Deduplication & Filter)...

🧼 LAPORAN PEMBERSIHAN AKHIR DATASET
🔹 Total baris awal audit           : 1,968,668
🔹 Baris setelah BOAB dibuang       : 1,645,204 (Berkurang 323,464)
🔹 Baris setelah duplikasi dibuang  : 1,001,973 (Berkurang 643,231)
🔹 Total Data Bersih Sempurna       : 1,001,973 Baris
📂 File final siap diboyong          : D:\indonesia_output\INDONESIA_STATION_INVENTORY_FINAL.csv


0

In [ ]:
import os
import gc
import time
import pandas as pd
import winsound
from obspy.clients.fdsn import Client
from obspy import UTCDateTime
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm

# --- CONFIG PATH (Menggunakan Format Windows Drive D:) ---
PATH_CSV_FINAL = r"D:\indonesia_output\INDONESIA_STATION_INVENTORY_FINAL.csv"
OUTPUT_WAVEFORM_DIR = r"D:\indonesia_output\output_indonesia"
LOG_ERROR_PATH = r"D:\indonesia_output\output_indonesia\log_error\error_log.csv"

# --- ACADEMIC IDENTITY ---
ACADEMIC_USER_AGENT = (
    "ResearchProject: Doctoral Dissertation in AI and Edge Computing; "
    "Researcher: Very Kurnia Bakti (Indonesia); "
    "Contact: verykurniabakti@gmail.com, verykurniabakti@harkatnegeri.ac.id; "
    "ID: Scopus:57209452703, Scholar:9-5-_dlAAAAJ, WOS:GPG-0207-2022"
)

clients = {
    "IRIS": Client("EARTHSCOPE", timeout=45, user_agent=ACADEMIC_USER_AGENT),
    "GFZ": Client("GFZ", timeout=45, user_agent=ACADEMIC_USER_AGENT)
}

def download_single_waveform(row_data):
    eid, time_str, net, sta, server = row_data
    
    try:
        t_event = UTCDateTime(time_str)
        year_folder = str(t_event.year)
        event_dir = os.path.join(OUTPUT_WAVEFORM_DIR, year_folder, str(eid))
        filename = f"{net}_{sta}_{eid}.mseed"
        file_path = os.path.join(event_dir, filename)
        
        # --- AUTO-RESUME TINGKAT FILE (WINDOWS SAFE) ---
        if os.path.exists(file_path) and os.path.getsize(file_path) > 0:
            return "SKIPPED", None
            
        # Pastikan folder tujuan otomatis terbuat
        os.makedirs(event_dir, exist_ok=True)
        
        cl = clients.get(server)
        if not cl:
            return "ERROR_SERVER", (eid, sta, "Server Config Error")
            
        # Windowing: 5 detik sebelum hingga 120 detik setelah event
        start_time = t_event - 5
        end_time = t_event + 120
        
        # Tarik data biner langsung dari server pusat
        st = cl.get_waveforms(network=net, station=sta, location="*", channel="BH*,HH*,EH*",
                              starttime=start_time, endtime=end_time)
        
        # --- DIRECT WRITE TO WINDOWS STORAGE ---
        st.write(file_path, format="MSEED")
        
        del st
        return "SUCCESS", None
            
    except Exception as e:
        err_name = type(e).__name__
        return f"FAILED_{err_name}", (eid, sta, err_name)

def run_waveform_download_pipeline():
    print("⚡ Starting Windows Direct Waveform Downloader Pipeline (Max-Velocity Mode)...")
    
    # Pastikan folder log_error sudah ada di Drive D:
    os.makedirs(os.path.dirname(LOG_ERROR_PATH), exist_ok=True)
    
    if not os.path.exists(PATH_CSV_FINAL):
        print(f"❌ Peta navigasi {PATH_CSV_FINAL} tidak ditemukan!")
        return
        
    print("⏳ Loading master navigation footprint into memory...")
    df = pd.read_csv(PATH_CSV_FINAL, usecols=['Event_ID', 'Time_UTC', 'Net', 'Station', 'Server'])
    tasks = list(df[['Event_ID', 'Time_UTC', 'Net', 'Station', 'Server']].itertuples(index=False, name=None))
    total_tasks = len(tasks)
    
    del df
    gc.collect()
    
    # Konfigurasi Thread untuk sistem operasi Windows
    MAX_WORKERS = 20 
    print(f"📡 Total Antrean: {total_tasks:,} file | Threads: {MAX_WORKERS} Workers")
    
    stats = {"SUCCESS": 0, "SKIPPED": 0, "FAILED": 0}
    error_logs = []
    last_beep = time.time()
    
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        pbar = tqdm(total=total_tasks, unit="file", desc="Downloading")
        
        for status, err_detail in executor.map(download_single_waveform, tasks):
            if status == "SUCCESS":
                stats["SUCCESS"] += 1
            elif status == "SKIPPED":
                stats["SKIPPED"] += 1
            else:
                stats["FAILED"] += 1
                if err_detail:
                    error_logs.append({
                        "Event_ID": err_detail[0],
                        "Station": err_detail[1],
                        "Error_Reason": err_detail[2]
                    })
                
            pbar.update(1)
            pbar.set_postfix({
                "Saved": stats["SUCCESS"], 
                "Resumed": stats["SKIPPED"], 
                "Failed": stats["FAILED"]
            })
            
            # --- PROTEKSI RAM OTOMATIS PERIODIK (Sangat krusial untuk Windows 8GB) ---
            if (stats["SUCCESS"] + stats["SKIPPED"] + stats["FAILED"]) % 500 == 0:
                gc.collect()
                if error_logs and len(error_logs) % 1000 == 0:
                    pd.DataFrame(error_logs).to_csv(LOG_ERROR_PATH, index=False)
            
            # Pengingat Audio berkala setiap 10 detik agar Bapak tahu skrip masih hidup
            if time.time() - last_beep > 10:
                winsound.Beep(600, 80)
                last_beep = time.time()

    pbar.close()
    
    # Simpan sisa log kesalahan download jika ada
    if error_logs:
        pd.DataFrame(error_logs).to_csv(LOG_ERROR_PATH, index=False)
        print(f"📝 Laporan eror download disimpan di: {LOG_ERROR_PATH}")
        
    print("\n" + "="*50)
    print("🏁 PIPELINE WINDOWS DIRECT DOWNLOAD SELESAI")
    print("="*50)
    print(f"✅ File Berhasil Ditulis ke Drive D: : {stats['SUCCESS']:,} file")
    print(f"🔄 File Dilewati (Auto-Resume)      : {stats['SKIPPED']:,} file")
    print(f"❌ File Gagal (Timeout/NoData)      : {stats['FAILED']:,} file")
    print(f"📂 Lokasi Penyimpanan                : {OUTPUT_WAVEFORM_DIR}")
    print("="*50)
    
    # Nada selebrasi selesai
    winsound.Beep(1200, 500)

if __name__ == "__main__":
    run_waveform_download_pipeline()